# DIA — API de productos y precios (España)

**Objetivo:** probar primero el endpoint real de búsqueda de DIA; extraer productos y precios; validar campos, precios unitarios y relevancia; exportar archivos en el esquema compartido con BM y Consum.

Fuente técnica: [adaptador DIA de grocery-cli](https://github.com/jgalea/grocery-cli/blob/main/internal/dia/dia.go), que documenta `GET /api/v1/search-back/search?q=...` y `search_items`. No es una API oficial para desarrolladores. El acceso puede cambiar. Los precios pueden corresponder a la región predeterminada de Madrid y no a cualquier tienda.

**IMPORTANTE:** `HTTP 200` no basta; el modo de prueba exige productos y precios reales. Las coincidencias de nombre son solo candidatas, no equivalencias confirmadas. No se utilizan claves ni cookies privadas. Las peticiones son secuenciales y limitadas.

In [1]:
# Si faltan paquetes: %pip install requests pandas
import re, time, json, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
import requests
import pandas as pd
from IPython.display import display

BASE_URL = "https://www.dia.es"
SEARCH_URL = BASE_URL + "/api/v1/search-back/search"
SEARCH_TERMS = ["leche", "huevos", "manzana", "plátano", "tomate", "arroz", "pasta", "pan", "pollo", "carne picada", "salmón", "yogur", "queso", "agua", "zumo", "atún", "garbanzos", "aceite", "detergente", "papel higiénico"]
TEST_MODE = True
TERMS = ["leche"] if TEST_MODE else SEARCH_TERMS
REQUEST_PAUSE = 1.0
TIMEOUT = 20
RETRIES = 2
HEADERS = {"Accept": "application/json", "User-Agent": "UC3M-WebAnalytics-StudentProject/1.0"}
ROOT = Path.cwd().parent if Path.cwd().name == "api_clients" else Path.cwd()
RAW_DIR, PROCESSED_DIR = ROOT / "data" / "raw", ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Modo:", "PRUEBA" if TEST_MODE else "EXTRACCIÓN COMPLETA", "| búsquedas:", TERMS)

Modo: PRUEBA | búsquedas: ['leche']


## 1. Conexión y estructura real del JSON

In [2]:
session = requests.Session()
session.headers.update(HEADERS)

def get_search(term):
    for attempt in range(RETRIES + 1):
        try:
            response = session.get(SEARCH_URL, params={"q": term}, timeout=TIMEOUT)
            if response.status_code in (429, 500, 502, 503, 504) and attempt < RETRIES:
                time.sleep(2 * (attempt + 1)); continue
            if not response.ok:
                raise RuntimeError(f"DIA HTTP {response.status_code}: {response.text[:250]}")
            data = response.json()
            if not isinstance(data, dict):
                raise ValueError("El endpoint no devolvió un objeto JSON")
            if not isinstance(data.get("search_items"), list):
                raise ValueError("No existe 'search_items' como lista. Claves: " + str(list(data)[:25]))
            return data
        except (requests.RequestException, ValueError) as exc:
            if attempt == RETRIES: raise
            print(f"Reintento de {term}: {type(exc).__name__}: {exc}")
            time.sleep(2 * (attempt + 1))

test_data = get_search("leche")
items = test_data["search_items"]
print("HTTP / JSON: OK")
print("Claves de respuesta:", list(test_data.keys()))
print("Productos recibidos:", len(items))
if not items:
    print("ATENCIÓN: la API respondió, pero no devolvió artículos.")
else:
    print("Claves del primer producto:", list(items[0].keys()))
    print("Primer producto (estructura abreviada):")
    print(json.dumps(items[0], ensure_ascii=False, indent=2)[:2500])

RuntimeError: DIA HTTP 403: <HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;dia&#46;es&#47;api&#47;v1&#47;search&#45;back&#47;search&#63;" on this server.<P>
Reference&#32;&#35;18&#

## 2. Normalización y verificación de calidad

DIA documenta `prices.price` (precio de venta) y `prices.price_per_unit`. **No asumimos la unidad**: si `prices.measure_unit` no permite reconocer €/kg, €/l, etc., conservamos el dato en bruto y no lo incorporamos como precio unitario válido. Tampoco inventamos precios anteriores u ofertas.

In [ ]:
UNIT_ALIASES = {"kg":"KILO", "kilo":"KILO", "kilogramo":"KILO", "g":"GRAMO", "gr":"GRAMO", "l":"LITRO", "litro":"LITRO", "ml":"MILILITRO", "cl":"CENTILITRO", "ud":"UNIDAD", "uds":"UNIDAD", "u":"UNIDAD", "unidad":"UNIDAD", "unidades":"UNIDAD", "docena":"DOCENA", "rollo":"ROLLO", "rollos":"ROLLO", "lavado":"LAVADO", "lavados":"LAVADO"}
FACTORS = {"GRAMO": ("KILO", 0.001), "MILILITRO": ("LITRO", .001), "CENTILITRO": ("LITRO", .01)}

def number(x):
    if x is None or isinstance(x, bool): return None
    try:
        v = float(str(x).replace("€", "").replace(",", ".").strip())
        return v if 0 <= v < 100000 else None
    except (ValueError, TypeError): return None

def unit_price_normalized(value, raw_unit):
    value = number(value)
    if value is None or not raw_unit: return None, None
    raw = str(raw_unit).lower().replace("€", "").replace("/", " ").strip()
    m = re.fullmatch(r"(\d+(?:[.,]\d+)?)?\s*([a-záéíóú]+)", raw)
    if not m: return None, None
    count = float(m.group(1).replace(",", ".")) if m.group(1) else 1.0
    label = UNIT_ALIASES.get(m.group(2))
    if not label or count <= 0: return None, None
    if label in FACTORS:
        target, factor = FACTORS[label]
        return round(value / (count * factor), 4), target
    return round(value / count, 4), label

def simplify(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', str(s).lower()) if not unicodedata.combining(ch))

# Filtro prudente: candidatos por nombre; la equivalencia entre supermercados exige más análisis.
# Términos negativos solo para la prueba de 'leche', para evitar aceptar aparatos y cosmética.
MILK_EXCLUSIONS = ('espumador', 'cazo', 'jabon', 'gel de', 'corporal', 'chocolate con leche', 'caramelo', 'galleta', 'crema de manos')
def relevant_candidate(term, name):
    t, name = simplify(term), simplify(name)
    tokens = t.split()
    matched = all(re.search(r'\b'+re.escape(token)+r's?\b', name) is not None for token in tokens)
    if t == 'leche' and any(x in name for x in MILK_EXCLUSIONS): return False
    return matched

def record(term, item):
    prices = item.get('prices') or {}
    if not isinstance(prices, dict): prices = {}
    price = number(prices.get('price'))
    raw_unit_price = number(prices.get('price_per_unit'))
    raw_unit = prices.get('measure_unit') or prices.get('unit') or item.get('measure_unit')
    unit_price, unit = unit_price_normalized(raw_unit_price, raw_unit)
    name = item.get('display_name') or item.get('name') or ''
    link = item.get('url') or ''
    img = item.get('image') or ''
    return {
        'searched_item': term, 'supermarket': 'DIA', 'product_id': str(item.get('object_id') or ''),
        'ean': item.get('ean'), 'brand': item.get('brand'), 'name': name,
        'category': item.get('category'), 'price': price,
        'regular_price': None, 'offer_price': None, 'on_offer': None,
        'price_per_unit': unit_price, 'unit': unit, 'unit_price_unit_type': raw_unit,
        'available': None, 'url': urljoin(BASE_URL, link) if link else None,
        'image_url': urljoin(BASE_URL, img) if img else None,
        'scraped_at': datetime.now(timezone.utc).isoformat(),
        'raw_price_per_unit': raw_unit_price, 'candidate_match': relevant_candidate(term, name),
        'has_valid_price': price is not None and price > 0,
        'has_valid_unit_price': unit_price is not None and unit_price > 0,
    }

preview = pd.DataFrame([record('leche', x) for x in items])
print('Productos encontrados:', len(preview))
if not preview.empty:
    display(preview[['name','price','raw_price_per_unit','unit_price_unit_type','price_per_unit','unit','candidate_match']].head(20))
    print('Candidatos por nombre:', int(preview.candidate_match.sum()))
    print('Candidatos con precio:', int((preview.candidate_match & preview.has_valid_price).sum()))
    print('Candidatos con precio por unidad:', int((preview.candidate_match & preview.has_valid_unit_price).sum()))

## 3. Extracción de búsquedas

El endpoint de búsqueda documentado devuelve `search_items`, pero **no hay paginación confirmada** en esta integración. No inventamos parámetros `page`/`offset`. Se registrará el tamaño recibido para decidir si necesitamos después un método de catálogo más completo.

In [ ]:
all_records, diagnostic = [], []
for idx, term in enumerate(TERMS, 1):
    try:
        data = test_data if term == 'leche' else get_search(term)
        products = data['search_items']
        rows = [record(term, x) for x in products if isinstance(x, dict)]
        all_records.extend(rows)
        diagnostic.append({'term': term, 'returned': len(products), 'candidates': sum(r['candidate_match'] for r in rows), 'with_price': sum(r['candidate_match'] and r['has_valid_price'] for r in rows), 'with_unit_price': sum(r['candidate_match'] and r['has_valid_unit_price'] for r in rows), 'error': None})
        print(f'[{idx}/{len(TERMS)}] {term}: {len(products)} resultados / {sum(r["candidate_match"] for r in rows)} candidatos')
    except Exception as exc:
        diagnostic.append({'term': term, 'returned': 0, 'candidates': 0, 'with_price': 0, 'with_unit_price': 0, 'error': str(exc)})
        print(f'[{idx}/{len(TERMS)}] ERROR {term}: {exc}')
    if idx < len(TERMS): time.sleep(REQUEST_PAUSE)

raw_df = pd.DataFrame(all_records)
diagnostic_df = pd.DataFrame(diagnostic)
display(diagnostic_df)

## 4. Exportación en el formato BM / Consum

- **Raw:** guarda todos los resultados extraídos y campos de diagnóstico.
- **Processed:** solo candidatos con precio, unidad e importe unitario verificables. Esto evita contaminar el optimizador.
- **Review:** candidatos excluidos del processed, para revisar manualmente.

La columna `on_offer` queda vacía (desconocida): no inferimos promociones de la diferencia entre valores sin evidencia.

In [ ]:
STANDARD_COLUMNS = ['searched_item','supermarket','product_id','ean','brand','name','category','price','regular_price','offer_price','on_offer','price_per_unit','unit','unit_price_unit_type','available','url','image_url','scraped_at']
# Separate test outputs from existing Selenium data
TEST_OUTPUT_DIR = ROOT / "data" / "api_validation" / "dia"
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_path = TEST_OUTPUT_DIR / "dia_products_raw.csv"
processed_path = TEST_OUTPUT_DIR / "dia_products.csv"
review_path = TEST_OUTPUT_DIR / "dia_products_review.csv"
quality_path = TEST_OUTPUT_DIR / "dia_quality_report.csv"

if raw_df.empty:
    print('SIN RESULTADOS. No se genera un CSV de catálogo vacío.')
else:
    raw_df = raw_df.drop_duplicates(subset=['searched_item','product_id','name'])
    valid = raw_df['candidate_match'] & raw_df['has_valid_price'] & raw_df['has_valid_unit_price']
    processed = raw_df.loc[valid, STANDARD_COLUMNS].drop_duplicates(subset=['product_id', 'searched_item']).reset_index(drop=True)
    review = raw_df.loc[~valid].copy()
    raw_df.to_csv(raw_path, index=False, encoding='utf-8-sig')
    processed.to_csv(processed_path, index=False, encoding='utf-8-sig')
    review.to_csv(review_path, index=False, encoding='utf-8-sig')
    diagnostic_df.to_csv(quality_path, index=False, encoding='utf-8-sig')
    print('RAW:', raw_path, len(raw_df))
    print('PROCESSED:', processed_path, len(processed))
    print('REVIEW:', review_path, len(review))
    print('QUALITY:', quality_path)
    if len(processed): display(processed.head(15))
    else: print('No hay productos que cumplan todos los criterios. Revisar raw y las unidades de DIA antes de ajustar filtros.')

## Interpretación y próximos pasos

1. Si la respuesta de prueba es `HTTP 200` con nombres y precios de alimentos, **la búsqueda funciona**.
2. Si `price_per_unit` está presente pero `measure_unit` falta, los precios unitarios se dejan sin normalizar, **sin inventar unidades**. Comprobar manualmente el JSON y usar categorías o ficha de producto para determinar las unidades.
3. Si devuelve pocos productos por búsqueda, **la cobertura no está demostrada**: la búsqueda actual no cuenta con paginación documentada.
4. Antes de optimizar cestas, validar código postal/zona de precios, surtido, disponibilidad y equivalencia por formato y características.

**Fuentes:** [adaptador `grocery-cli` DIA](https://github.com/jgalea/grocery-cli/blob/main/internal/dia/dia.go); [extracción de categorías DIA en otro proyecto](https://github.com/joseluam97/Supermarket-Price-Scraper/blob/main/supermarket/dia.py).